In [9]:
# !pip install azure-ai-formrecognizer
# !pip install azure-core
# #

In [17]:
# Imports & Azure connection
import os
import re
import sqlite3
from azure.ai.formrecognizer import DocumentAnalysisClient
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv
load_dotenv() 
ENDPOINT = os.getenv("endpoint_d")
KEY = os.getenv("key_d")

client = DocumentAnalysisClient(
    endpoint=ENDPOINT,
    credential=AzureKeyCredential(KEY)
)

DB = "student_cards.db"

In [11]:
# Create Database Table
def create_table():

    conn = sqlite3.connect(DB)

    conn.execute("""
    CREATE TABLE IF NOT EXISTS student_id_cards(

    id INTEGER PRIMARY KEY AUTOINCREMENT,

    student_name TEXT,
    father_name TEXT,
    dob TEXT,
    class TEXT,
    section TEXT,
    roll_no TEXT,
    mobile TEXT,
    academic_year TEXT,
    source_file TEXT

    )
    """)

    conn.close()

create_table()

print("table created")

table created


In [12]:
def extract_fields_from_json(result):

    data = {
        "student_name": None,
        "father_name": None,
        "dob": None,
        "class": None,
        "section": None,
        "roll_no": None,
        "mobile": None,
        "academic_year": None
    }

    for kv in result.key_value_pairs:

        key = kv.key.content.lower() if kv.key else ""
        value = kv.value.content if kv.value else ""

        if "name" in key and "father" not in key:
            data["student_name"] = value

        elif "father" in key:
            data["father_name"] = value

        elif "dob" in key:
            data["dob"] = value

        elif "class" in key:
            data["class"] = value

        elif "sec" in key:
            data["section"] = value

        elif "roll" in key or "id" in key or "rod" in key:
            data["roll_no"] = value

        elif "mobile" in key:
            data["mobile"] = value

    # extract academic year from full text
    import re
    match = re.search(r"\d{4}-\d{4}", result.content)

    if match:
        data["academic_year"] = match.group()

    return data

In [13]:
#Image processing
def process_image(path):

    with open(path, "rb") as f:

        poller = client.begin_analyze_document(
            "prebuilt-document",   # important change
            f
        )

    result = poller.result()

    return extract_fields_from_json(result)

In [14]:
# Store Data in SQLite
def save_db(data,file):

    conn = sqlite3.connect(DB)

    conn.execute("""
    INSERT INTO student_id_cards(

    student_name,
    father_name,
    dob,
    class,
    section,
    roll_no,
    mobile,
    academic_year,
    source_file

    )

    VALUES (?,?,?,?,?,?,?,?,?)
    """,

    (

    data.get("student_name"),
    data.get("father_name"),
    data.get("dob"),
    data.get("class"),
    data.get("section"),
    data.get("roll_no"),
    data.get("mobile"),
    data.get("year"),
    file

    ))

    conn.commit()
    conn.close()

In [16]:
# Run Pipeline
folder = "cards"

for file in os.listdir(folder):

    path = os.path.join(folder,file)

    data = process_image(path)
    print(data)

    # save_db(data,file)

    print("processed:",file)

{'student_name': 'Raju', 'father_name': 'Matu', 'dob': '05-02-2000', 'class': 'X', 'section': '4 B', 'roll_no': '. 02', 'mobile': '002244668', 'academic_year': '2020-2021'}
processed: img1.jpeg
{'student_name': None, 'father_name': 'Manu', 'dob': '05-02-2000', 'class': 'X', 'section': 'B', 'roll_no': '245860', 'mobile': '002244668', 'academic_year': None}
processed: img2.jpeg
{'student_name': 'Raju', 'father_name': 'Manu', 'dob': None, 'class': '05-02-2000\nX', 'section': 'B', 'roll_no': '02', 'mobile': '002244068', 'academic_year': '2020-2021'}
processed: img3.jpeg
{'student_name': None, 'father_name': 'Manu', 'dob': '05-02-2000', 'class': ':selected:', 'section': 'B', 'roll_no': '245860', 'mobile': '002244668', 'academic_year': None}
processed: img4.jpeg
